Purpose: Look at results of updated, corrected polynomial modeling on three-way shared TSGs across physiotypes and pathway genes (CAM genes, photorespiration, N-metabolism, citrate, and clock genes).<br>
Author: Anna Pardo<br>
Date initiated: July 23, 2026

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import os
import json

In [23]:
# load results (modeling & post-hoc)
modres = pd.read_csv("./polymod3_modelres_fullmod_TSGs_pathwaygenes.txt",sep="\t",header="infer")
phres = pd.read_csv("./polymod3_posthocres_TSGs_pathwaygenes.txt",sep="\t",header="infer")
modres.head()

,Sum Sq,Df,F value,Pr(>F),Factors,GeneID,FDR_p
0,0.465785,1,1.333161,2.485871e-01,(Intercept),Yucal.11G006100.v2.1,3.590887e-01
1,147.216687,3,140.453660,3.730225e-73,"poly(ZT, 3)",Yucal.11G006100.v2.1,3.503564e-71
2,0.757162,1,2.167136,1.413804e-01,treat,Yucal.11G006100.v2.1,2.276692e-01
3,2.328701,2,3.332583,3.619521e-02,phys,Yucal.11G006100.v2.1,7.202747e-02
4,0.015806,3,0.015080,9.974731e-01,"poly(ZT, 3):treat",Yucal.11G006100.v2.1,9.992700e-01


In [4]:
modres["Factors"].unique()

array(['(Intercept)', 'poly(ZT, 3)', 'treat', 'phys', 'poly(ZT, 3):treat',
       'poly(ZT, 3):phys', 'treat:phys', 'poly(ZT, 3):treat:phys',
       'Residuals'], dtype=object)

In [3]:
phres.head()

,Comparison,Estimate,SE,t.value,p.value,GeneID,FDR_p
0,CAM - C3+CAM,-0.161012,0.070045,-2.298688,5.533985e-02,Yucal.11G006100.v2.1,0.155789
1,facultative CAM - C3+CAM,-0.149852,0.086278,-1.736850,1.887997e-01,Yucal.11G006100.v2.1,0.396686
2,facultative CAM - CAM,0.011160,0.095818,0.116472,9.924354e-01,Yucal.11G006100.v2.1,1.000000
3,CAM - C3+CAM,-0.078946,0.090571,-0.871650,6.545220e-01,Yucal.15G098200.v2.1,0.915452
4,facultative CAM - C3+CAM,-0.632811,0.111560,-5.672380,3.893534e-08,Yucal.15G098200.v2.1,0.000001


In [24]:
# load pathway & general gene name annotations
pathgenes = pd.read_csv("./photresp_N_citrate_genes_Yucca.csv",sep=",",header="infer")
pathgenes = pathgenes[pathgenes["Pathway"].isin(["PhotResp","N-metab","CA-cycle"])]
pathgenes.head()

,Pathway,gene_family,GeneID,subgenome,gene_name_unique
0,PhotResp,PGP,Yucal.01G302600.v2.1,Ya,Ya_PGP_1
1,PhotResp,PGP,Yucal.22G052300.v2.1,Ya,Ya_PGP_2
2,PhotResp,PGP,YufilH1006999m.g,Yf,Yf_PGP_1
3,PhotResp,PGP,YufilH1040717m.g,Yf,Yf_PGP_2
4,PhotResp,PGP,YufilH1084368m.g,Yf,Yf_PGP_3


In [25]:
camgenes = pd.read_csv("./camgenes_Ya_Yf_orthology_synteny.txt",sep="\t",header="infer")
clockgenes = pd.read_csv("./circadian_light_genes_by_orthology_Yucca.csv",sep=",",header="infer")

In [26]:
# load overall gene names annotation
gannot = pd.read_csv("./gene_annotations/Yg_bothsg_1geneperrow_genefunctions.txt",sep="\t",header="infer")
gannot.head()

,GeneID,syntelogID,Description
0,Yucal.01G000100.v2.1,recip_syn1,cwf21 domain
1,YufilH1000007m.g,recip_syn2,"DnaJ homolog subfamily A member 2, DNA repair,..."
2,Yucal.01G000200.v2.1,recip_syn2,Endonuclease V
3,YufilH1000011m.g,recip_syn3,"nan, Uncharacterized high-glucose-regulated pr..."
4,Yucal.01G000400.v2.1,recip_syn3,YT521-B-like domain


In [10]:
pathgenes.head()

,Pathway,gene_family,GeneID,subgenome,gene_name_unique
0,PhotResp,PGP,Yucal.01G302600.v2.1,Ya,Ya_PGP_1
1,PhotResp,PGP,Yucal.22G052300.v2.1,Ya,Ya_PGP_2
2,PhotResp,PGP,YufilH1006999m.g,Yf,Yf_PGP_1
3,PhotResp,PGP,YufilH1040717m.g,Yf,Yf_PGP_2
4,PhotResp,PGP,YufilH1084368m.g,Yf,Yf_PGP_3


In [12]:
clockgenes.head()

,GeneID,Orthogroup,gene_name,subgenome,gene_name_unique
0,Yucal.11G006100.v2.1,OG0002937,TOC1,Ya,Ya_TOC1_1
1,Yucal.15G098200.v2.1,OG0002937,TOC1,Ya,Ya_TOC1_2
2,Yucal.16G106200.v2.1,OG0002937,TOC1,Ya,Ya_TOC1_3
3,YufilH1057026m.g,OG0002937,TOC1,Yf,Yf_TOC1_1
4,YufilH1057027m.g,OG0002937,TOC1,Yf,Yf_TOC1_2


In [27]:
camgenes["Pathway"] = "CAM"
clockgenes["Pathway"] = "Clock"
all_pathways = pd.concat([
    pathgenes,
    camgenes.rename(columns={"gene_name":"gene_family","gene_abbr_unique":"gene_name_unique"})[["Pathway","gene_family",
                                                                                               "GeneID","subgenome",
                                                                                               "gene_name_unique"]],
    clockgenes.rename(columns={"gene_name":"gene_family"})[["Pathway","gene_family","GeneID","subgenome","gene_name_unique"]]
])
all_pathways.head()

,Pathway,gene_family,GeneID,subgenome,gene_name_unique
0,PhotResp,PGP,Yucal.01G302600.v2.1,Ya,Ya_PGP_1
1,PhotResp,PGP,Yucal.22G052300.v2.1,Ya,Ya_PGP_2
2,PhotResp,PGP,YufilH1006999m.g,Yf,Yf_PGP_1
3,PhotResp,PGP,YufilH1040717m.g,Yf,Yf_PGP_2
4,PhotResp,PGP,YufilH1084368m.g,Yf,Yf_PGP_3


In [15]:
gannot.head()

,GeneID,syntelogID,Description
0,Yucal.01G000100.v2.1,recip_syn1,cwf21 domain
1,YufilH1000007m.g,recip_syn2,"DnaJ homolog subfamily A member 2, DNA repair,..."
2,Yucal.01G000200.v2.1,recip_syn2,Endonuclease V
3,YufilH1000011m.g,recip_syn3,"nan, Uncharacterized high-glucose-regulated pr..."
4,Yucal.01G000400.v2.1,recip_syn3,YT521-B-like domain


In [22]:
# add gene info to modres & phres: make a function to do this cleanly
def add_gene_info(resdf):
    pathres = resdf.merge(all_pathways,how="inner")
    nonpath = resdf.merge(gannot,how="left")
    return pathres.merge(nonpath,how="outer")

In [28]:
mres_annot = add_gene_info(modres)
pres_annot = add_gene_info(phres)

In [29]:
# find the significant model results
sigmod = mres_annot[mres_annot["FDR_p"]<0.05]

In [30]:
len(mres_annot["GeneID"].unique()) - len(sigmod["GeneID"].unique())

14

In [33]:
len(sigmod["GeneID"].unique())

4600

## Update summary table of all results with polynomial modeling & post-hoc results

In [34]:
sumtbl = pd.read_csv("./pathway_genes_results_summary.csv",sep=",",header="infer")
sumtbl.head()

,Pathway,GeneID,Gene Family,Parental Origin,Gene Name,TS in Ya,TS in Yf,TS in CAM,TS in C3+CAM,TS in facCAM,...,facultative CAM_D,polyZT*treat*phys,polyZT*phys,polyZT*treat,ASE_C3+CAM_W,ASE_C3+CAM_D,ASE_facultative CAM_W,ASE_CAM_W,ASE_facultative CAM_D,ASE_CAM_D
0,CAM-dark,Yucal.01G165600.v2.1,bCA1234,Ya,Ya_bCA1234_1,Y,N,N,N,N,...,UP,sig,no result,sig,"Ya, 10.0%","Ya, 40.0%; Yf, 10.0%","Ya, 20.0%","Ya, 16.7%",no result,"Ya, 16.7%"
1,CAM-dark,Yucal.02G112700.v2.1,bCA1234,Ya,Ya_bCA1234_2,Y,N,N,N,N,...,ELD_P2,not sig,no result,not sig,"Yf, 100.0%","Yf, 90.0%","Yf, 100.0%","Yf, 83.3%","Yf, 60.0%","Yf, 83.3%"
2,CAM-dark,Yucal.04G001000.v2.1,bCA5,Ya,Ya_bCA5_1,N,N,Y,N,N,...,"DOWN, ELD_P2, ADD",sig,sig,sig,"Ya, 70.0%","Ya, 60.0%; Yf, 20.0%","Ya, 80.0%","Ya, 100.0%","Ya, 40.0%","Ya, 83.3%"
3,CAM-dark,Yucal.07G000800.v2.1,bCA5,Ya,Ya_bCA5_2,Y,N,N,N,N,...,"ELD_P1, ADD",sig,sig,sig,"Ya, 100.0%","Ya, 90.0%","Ya, 100.0%","Ya, 83.3%","Ya, 60.0%","Ya, 83.3%"
4,CAM-dark,Yucal.03G120900.v2.1,NAD-MDH-cp,Ya,Ya_NAD-MDH-cp_1,Y,N,N,N,N,...,DOWN,not sig,no result,not sig,"Ya, 50.0%","Ya, 50.0%","Ya, 60.0%","Ya, 33.3%","Ya, 40.0%","Ya, 33.3%"


In [35]:
# drop prior polymod columns
sumtbl = sumtbl.drop(["polyZT*treat*phys","polyZT*phys","polyZT*treat"],axis=1)
sumtbl.head()

,Pathway,GeneID,Gene Family,Parental Origin,Gene Name,TS in Ya,TS in Yf,TS in CAM,TS in C3+CAM,TS in facCAM,...,C3+CAM_W,CAM_W,facultative CAM_W,facultative CAM_D,ASE_C3+CAM_W,ASE_C3+CAM_D,ASE_facultative CAM_W,ASE_CAM_W,ASE_facultative CAM_D,ASE_CAM_D
0,CAM-dark,Yucal.01G165600.v2.1,bCA1234,Ya,Ya_bCA1234_1,Y,N,N,N,N,...,"ADD, ELD_P1, UP","UP, ELD_P2","ELD_P1, UP",UP,"Ya, 10.0%","Ya, 40.0%; Yf, 10.0%","Ya, 20.0%","Ya, 16.7%",no result,"Ya, 16.7%"
1,CAM-dark,Yucal.02G112700.v2.1,bCA1234,Ya,Ya_bCA1234_2,Y,N,N,N,N,...,"ADD, ELD_P2","ELD_P2, ADD, UP",ELD_P2,ELD_P2,"Yf, 100.0%","Yf, 90.0%","Yf, 100.0%","Yf, 83.3%","Yf, 60.0%","Yf, 83.3%"
2,CAM-dark,Yucal.04G001000.v2.1,bCA5,Ya,Ya_bCA5_1,N,N,Y,N,N,...,"DOWN, ELD_P2, UP","ELD_P1, ELD_P2","ELD_P2, DOWN, ELD_P1","DOWN, ELD_P2, ADD","Ya, 70.0%","Ya, 60.0%; Yf, 20.0%","Ya, 80.0%","Ya, 100.0%","Ya, 40.0%","Ya, 83.3%"
3,CAM-dark,Yucal.07G000800.v2.1,bCA5,Ya,Ya_bCA5_2,Y,N,N,N,N,...,"DOWN, UP",DOWN,DOWN,"ELD_P1, ADD","Ya, 100.0%","Ya, 90.0%","Ya, 100.0%","Ya, 83.3%","Ya, 60.0%","Ya, 83.3%"
4,CAM-dark,Yucal.03G120900.v2.1,NAD-MDH-cp,Ya,Ya_NAD-MDH-cp_1,Y,N,N,N,N,...,"DOWN, ELD_P2","DOWN, ELD_P2",DOWN,DOWN,"Ya, 50.0%","Ya, 50.0%","Ya, 60.0%","Ya, 33.3%","Ya, 40.0%","Ya, 33.3%"


In [36]:
mres_annot["Factors"].unique()

array(['(Intercept)', 'poly(ZT, 3)', 'treat', 'phys', 'poly(ZT, 3):treat',
       'poly(ZT, 3):phys', 'treat:phys', 'poly(ZT, 3):treat:phys',
       'Residuals'], dtype=object)

In [66]:
phres.head()

,Comparison,Estimate,SE,t.value,p.value,GeneID,FDR_p
0,CAM - C3+CAM,-0.161012,0.070045,-2.298688,5.533985e-02,Yucal.11G006100.v2.1,0.155789
1,facultative CAM - C3+CAM,-0.149852,0.086278,-1.736850,1.887997e-01,Yucal.11G006100.v2.1,0.396686
2,facultative CAM - CAM,0.011160,0.095818,0.116472,9.924354e-01,Yucal.11G006100.v2.1,1.000000
3,CAM - C3+CAM,-0.078946,0.090571,-0.871650,6.545220e-01,Yucal.15G098200.v2.1,0.915452
4,facultative CAM - C3+CAM,-0.632811,0.111560,-5.672380,3.893534e-08,Yucal.15G098200.v2.1,0.000001


In [70]:
# set up columns for significance of all factors
factdict = {"GeneID":[]}
for i in mres_annot["Factors"].unique():
    if i!="Residuals":
        factdict[i] = []

In [71]:
# add post-hoc information columns
for i in phres["Comparison"].unique():
    factdict[i] = []

In [73]:
# populate the dictionary
for i in sumtbl["GeneID"].unique():
    if i in list(modres["GeneID"]):
        factdict["GeneID"].append(i)
        moddf = modres[modres["GeneID"]==i]
        phdf = phres[phres["GeneID"]==i]
        for k,v in factdict.items():
            if k in list(modres["Factors"]):
                #print(k)
                pval = moddf.loc[moddf['Factors'] == k, 'FDR_p'].iloc[0]
                if pval < 0.05:
                    v.append("significant")
                else:
                    v.append("not significant")
            elif k in list(phres["Comparison"]):
                #print(k)
                pval = phdf.loc[phdf["Comparison"]==k,"FDR_p"].iloc[0]
                if pval < 0.05:
                    v.append("significant")
                else:
                    v.append("not significant")

In [77]:
sumtbl_withstats = sumtbl.merge(pd.DataFrame(factdict),how="left")
sumtbl_withstats.head()

,Pathway,GeneID,Gene Family,Parental Origin,Gene Name,TS in Ya,TS in Yf,TS in CAM,TS in C3+CAM,TS in facCAM,...,"poly(ZT, 3)",treat,phys,"poly(ZT, 3):treat","poly(ZT, 3):phys",treat:phys,"poly(ZT, 3):treat:phys",CAM - C3+CAM,facultative CAM - C3+CAM,facultative CAM - CAM
0,CAM-dark,Yucal.01G165600.v2.1,bCA1234,Ya,Ya_bCA1234_1,Y,N,N,N,N,...,significant,significant,significant,not significant,not significant,not significant,not significant,not significant,significant,not significant
1,CAM-dark,Yucal.02G112700.v2.1,bCA1234,Ya,Ya_bCA1234_2,Y,N,N,N,N,...,not significant,significant,not significant,not significant,not significant,not significant,not significant,not significant,not significant,not significant
2,CAM-dark,Yucal.04G001000.v2.1,bCA5,Ya,Ya_bCA5_1,N,N,Y,N,N,...,not significant,significant,significant,not significant,not significant,significant,not significant,significant,not significant,significant
3,CAM-dark,Yucal.07G000800.v2.1,bCA5,Ya,Ya_bCA5_2,Y,N,N,N,N,...,significant,not significant,significant,not significant,significant,not significant,not significant,significant,significant,not significant
4,CAM-dark,Yucal.03G120900.v2.1,NAD-MDH-cp,Ya,Ya_NAD-MDH-cp_1,Y,N,N,N,N,...,significant,significant,not significant,not significant,not significant,not significant,not significant,not significant,not significant,not significant


In [78]:
sumtbl_withstats.to_csv("./pathway_genes_results_updatedpolymod.csv",sep=",",header=True,index=False)